<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/Decoder_Only_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)

model_id = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

In [ ]:
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
next_token_logits = logits[0, -1, :]

probs = torch.softmax(next_token_logits, dim=-1)

top_k = 10
top_probs, top_ids = torch.topk(probs, top_k)

for prob, token_id in zip(top_probs, top_ids):
    token = tokenizer.decode(token_id)
    print(f"{token!r}: {prob.item():.4f}")

#Greedy decoding

In [ ]:
next_token_id = torch.argmax(probs).item()
next_token = tokenizer.decode(next_token_id)

print("Selected token:", repr(next_token))
print("New text:", prompt + next_token)

In [ ]:
def manual_greedy(prompt, steps=20):
    text = prompt

    for _ in range(steps):
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[0, -1, :]
        probs = torch.softmax(logits, dim=-1)

        next_token_id = torch.argmax(probs).unsqueeze(0)
        next_token = tokenizer.decode(next_token_id)

        text += next_token

    return text

In [ ]:
print(manual_greedy("Once upon a time", steps=30))

#Sampling

In [ ]:
def manual_sampling(prompt, steps=20):
    text = prompt

    for _ in range(steps):
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[0, -1, :]
        probs = torch.softmax(logits, dim=-1)

        next_token_id = torch.multinomial(probs, num_samples=1)
        next_token = tokenizer.decode(next_token_id)

        text += next_token

    return text

In [ ]:
print(manual_sampling("Once upon a time", steps=30))

#temperature, top-k, top-p

In [ ]:
# probs = torch.softmax(logits)
# probs = softmax(logits / temperature)

In [ ]:
def manual_temperature_sampling(prompt, steps=20, temperature=0.7):
    text = prompt

    for _ in range(steps):
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[0, -1, :]

        scaled_logits = logits / temperature
        probs = torch.softmax(scaled_logits, dim=-1)

        next_token_id = torch.multinomial(probs, num_samples=1)
        next_token = tokenizer.decode(next_token_id)

        text += next_token

    return text

In [ ]:
for temp in [0.3, 0.7, 1.2]:
    print("Temperature:", temp)
    print(manual_temperature_sampling("Once upon a time", steps=30, temperature=temp))

#Top-k sampling

In [ ]:
def manual_top_k_sampling(prompt, steps=20, top_k=50, temperature=0.7):
    text = prompt

    for _ in range(steps):
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[0, -1, :]
        logits = logits / temperature

        top_values, top_indices = torch.topk(logits, top_k)

        top_probs = torch.softmax(top_values, dim=-1)

        sampled_index = torch.multinomial(top_probs, num_samples=1)
        next_token_id = top_indices[sampled_index]

        next_token = tokenizer.decode(next_token_id)
        text += next_token

    return text

In [ ]:
for k in [5, 20, 50]:
    print("Top-k:", k)
    print(manual_top_k_sampling("Once upon a time", steps=30, top_k=k, temperature=0.8))

#Top-p / nucleus sampling

In [ ]:
def manual_top_p_sampling(prompt, steps=20, top_p=0.9, temperature=0.7):
    text = prompt

    for _ in range(steps):
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[0, -1, :]
        logits = logits / temperature

        probs = torch.softmax(logits, dim=-1)

        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        mask = cumulative_probs <= top_p
        mask[0] = True

        filtered_probs = sorted_probs[mask]
        filtered_indices = sorted_indices[mask]

        filtered_probs = filtered_probs / filtered_probs.sum()

        sampled_index = torch.multinomial(filtered_probs, num_samples=1)
        next_token_id = filtered_indices[sampled_index]

        next_token = tokenizer.decode(next_token_id)
        text += next_token

    return text

In [ ]:
for p in [0.5, 0.8, 0.95]:
    print("Top-p:", p)
    print(manual_top_p_sampling("Once upon a time", steps=40, top_p=p, temperature=0.8))

#Hugging Face generate()

In [ ]:
def hf_generate(prompt, max_new_tokens=80, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        **kwargs
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = "Artificial intelligence is"

In [ ]:
output = hf_generate(
    prompt,
    do_sample=False
)

print(repr(output))

In [ ]:
print(hf_generate(
    prompt,
    do_sample=True
))

In [ ]:
print(hf_generate(
    prompt,
    do_sample=True,
    temperature=0.7
))

In [ ]:
print(hf_generate(
    prompt,
    do_sample=True,
    top_k=50,
    temperature=0.8
))

In [ ]:
print(hf_generate(
    prompt,
    do_sample=True,
    top_p=0.9,
    temperature=0.7
))

In [ ]:
print(hf_generate(
    prompt,
    do_sample=False,
    num_beams=5,
    early_stopping=True
))

# CHAT-BOT

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

chat_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_id)
chat_model = AutoModelForCausalLM.from_pretrained(
    chat_model_id,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
def chat(user_message, history=None):
    if history is None:
        history = []

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI teacher. Explain concepts simply and step by step."
        }
    ]

    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_message})

    text = chat_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = chat_tokenizer(text, return_tensors="pt").to(chat_model.device)

    outputs = chat_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        top_p=0.9,
        temperature=0.7
    )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = chat_tokenizer.decode(generated_ids, skip_special_tokens=True)

    return answer.strip()

In [ ]:
history = []

question = "What is causal masking in decoder-only models?"
answer = chat(question, history)

print("User:", question)
print("Bot:", answer)

history.append((question, answer))

In [ ]:
history = []

while True:
    user_message = input("You: ")

    if user_message.lower() in ["exit", "quit", "stop"]:
        break

    answer = chat(user_message, history)
    print("Bot:", answer)

    history.append((user_message, answer))